# BotHP (KDD 2025) on MGTAB

Repo: Peien429/BotHP, first set up by Phúc. Run this in its **own** Colab runtime with a GPU: BotHP pins torch 1.13, so it uses a Python 3.10 environment made with `uv`.

Two runs, 5 seeds each (BotHP's default seeds), about 1 hour each:

- **A. BotHP's own split** (`process.py`: random 70/20/10, seed 0; follower + friend). Checks the published numbers. This split includes our test accounts, so it is never used to tune our models.
- **B. Our split:** the same graph and settings, but with our `split()`. BotHP selects the epoch on its `val_mask` and reports on its `test_mask`, so here **both are our validation set**. Our test set is not used.

**Patches (no model change):** BotHP's evaluation also computes **macro-F1** and **PR-AUC** from the same predictions. BotHP's own F1 is binary F1 of the bot class, and it predicts with argmax (threshold 0.5, not tuned).

**Note:** BotHP's `process.py` swaps the masks on purpose: its `test_mask` is the 20% block and its `val_mask` is the 10% block.

At the end, download the zip and the notebook (File > Download > .ipynb) and commit both.

In [ ]:
import os, re, sys, glob, shutil, subprocess, zipfile
import numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "Turn on a GPU: Runtime > Change runtime type"
TEAM, BOTHP, PY = "/content/sna-content-moderation", "/content/BotHP", "/content/venv310/bin/python"
%cd /content
!pip install -q uv gdown
!test -d BotHP || git clone -q https://github.com/Peien429/BotHP.git
!test -d sna-content-moderation || git clone -q https://github.com/ThSonDev/sna-content-moderation.git
!cd BotHP && git log -1 --format="BotHP repo commit %h (%ad)"
os.makedirs(f"{TEAM}/results/logs", exist_ok=True)

/content
BotHP repo commit f00600a (Sun May 25 23:05:14 2025 +0800)


## 1. Python 3.10 environment (tested by Phúc; prebuilt wheels only)

In [ ]:
%cd /content/BotHP
!rm -rf /content/venv310 && uv venv -q --python 3.10 /content/venv310
!uv pip install -q --python {PY} torch==1.13.0 torchvision==0.14.0 torchaudio==0.13.0 --extra-index-url https://download.pytorch.org/whl/cu117 --index-strategy unsafe-best-match
!uv pip install -q --python {PY} --only-binary :all: --no-deps --find-links https://data.pyg.org/whl/torch-1.13.0+cu117.html torch-scatter torch-sparse torch-cluster torch-spline-conv
# requirements.txt without mkl-service and the torch packages above; faiss is missing from it (models/pretrain_model.py needs it)
!grep -v -E "^(mkl-service|torch==|torch-cluster|torch-scatter|torch-sparse|torch-spline-conv|torchaudio|torchvision)" requirements.txt > req_rest.txt
!uv pip install -q --python {PY} -r req_rest.txt faiss-cpu==1.7.4
!{PY} -c "import torch, torch_geometric, faiss; print(torch.__version__, torch_geometric.__version__, faiss.__version__, torch.cuda.is_available())"

/content/BotHP
1.13.0+cu117 2.1.0 1.7.4 True


## 2. Data and patches

In [ ]:
# Team copy of MGTAB -> dataset/raw/MGTAB. process.py reads label.pt, so labels_bot.pt is copied to that name.
RAW = f"{BOTHP}/dataset/raw/MGTAB"
if not os.path.exists(f"{RAW}/features.pt"):
    subprocess.run(["gdown", "1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe", "-O", "/content/MGTAB.zip"], check=True)
    zipfile.ZipFile("/content/MGTAB.zip").extractall("/content/_mgtab")
    hit = glob.glob("/content/_mgtab/**/features.pt", recursive=True)
    assert hit, "features.pt not found in the zip"
    for f in glob.glob(os.path.dirname(hit[0]) + "/*.pt"):
        shutil.copy(f, RAW)
shutil.copy(f"{RAW}/labels_bot.pt", f"{RAW}/label.pt")
print(sorted(os.listdir(RAW)))

def patch(path, old, new):
    s = open(path).read()
    if new in s:
        return
    assert s.count(old) == 1, f"patch target not found in {path}: {old!r}"
    open(path, "w").write(s.replace(old, new))

ev = f"{BOTHP}/node_classification_evaluation.py"
patch(ev, "precision_score, recall_score, precision_recall_curve",
          "precision_score, recall_score, precision_recall_curve, average_precision_score")
patch(ev, "        'recall': 0.0,\n", "        'recall': 0.0,\n        'macro_f1': 0.0,\n        'pr_auc': 0.0,\n")
patch(ev, "        metrics['recall'] = round(recall_score(y_true, y_pred_label), 5)\n",
          "        metrics['recall'] = round(recall_score(y_true, y_pred_label), 5)\n"
          "        metrics['macro_f1'] = round(f1_score(y_true, y_pred_label, average='macro'), 5)\n"
          "        metrics['pr_auc'] = round(average_precision_score(y_true, y_pred_score), 5)\n")
patch(f"{BOTHP}/main_transductive.py", "for i in ['accuracy', 'precision', 'recall', 'f1']:",
      "for i in ['accuracy', 'precision', 'recall', 'f1', 'macro_f1', 'pr_auc']:")
!grep -n "macro_f1\|pr_auc" {ev} main_transductive.py

['edge_index.pt', 'edge_type.pt', 'edge_weight.pt', 'features.pt', 'label.pt', 'labels_bot.pt', 'labels_stance.pt', 'process.py']
/content/BotHP/node_classification_evaluation.py:15:        'macro_f1': 0.0,
/content/BotHP/node_classification_evaluation.py:16:        'pr_auc': 0.0,
/content/BotHP/node_classification_evaluation.py:36:        metrics['macro_f1'] = round(f1_score(y_true, y_pred_label, average='macro'), 5)
/content/BotHP/node_classification_evaluation.py:37:        metrics['pr_auc'] = round(average_precision_score(y_true, y_pred_score), 5)
main_transductive.py:115:    for i in ['accuracy', 'precision', 'recall', 'f1', 'macro_f1', 'pr_auc']:


In [ ]:
RESULTS = []

def run_bothp(name, split_note, eval_set):
    """Runs BotHP (5 default seeds) on dataset/processed/MGTAB-followers-friends.pt; saves the log; parses the summary."""
    log = f"{TEAM}/results/logs/bothp_{name}.log"
    with open(log, "w") as f:
        subprocess.run([PY, "main_transductive.py", "--dataset", "MGTAB-FF"], cwd=BOTHP, stdout=f,
                       stderr=subprocess.STDOUT, check=True)
    print("".join(l for l in open(log) if l.startswith(("final_", "early-stopping_"))))
    found = re.findall(r"^(final|early-stopping)_(\w+): ([\d.]+)±([\d.]+)", open(log).read(), re.M)
    assert len(found) == 12, f"expected 12 summary lines, found {len(found)}; check {log}"
    for kind in ["final", "early-stopping"]:
        m = {k: (float(a), float(b)) for t, k, a, b in found if t == kind}
        RESULTS.append(dict(method=f"BotHP ({kind} epoch)", split=split_note, relations="follower, friend",
                            eval_set=eval_set, runs=5, accuracy=m["accuracy"][0], accuracy_std=m["accuracy"][1],
                            macro_f1=m["macro_f1"][0], macro_f1_std=m["macro_f1"][1],
                            binary_f1=m["f1"][0], binary_f1_std=m["f1"][1], pr_auc=m["pr_auc"][0],
                            pr_auc_std=m["pr_auc"][1], note="argmax prediction (threshold 0.5, not tuned)"))
    return pd.DataFrame(RESULTS)

## 3. Run A: BotHP's own split (check the published numbers)

In [ ]:
%cd /content/BotHP/dataset/raw/MGTAB
!{PY} process.py
run_bothp("own_split", "BotHP process.py: random 70/20/10, seed 0", "BotHP test_mask (20% block)")

/content/BotHP/dataset/raw/MGTAB
final_accuracy: 0.9101±0.0014
early-stopping_accuracy: 0.9102±0.0012
final_precision: 0.8514±0.0063
early-stopping_precision: 0.8457±0.0042
final_recall: 0.8048±0.0135
early-stopping_recall: 0.8128±0.0100
final_f1: 0.8273±0.0046
early-stopping_f1: 0.8289±0.0035
final_macro_f1: 0.8833±0.0027
early-stopping_macro_f1: 0.8840±0.0021
final_pr_auc: 0.8869±0.0038
early-stopping_pr_auc: 0.8878±0.0034



,method,split,relations,eval_set,runs,accuracy,accuracy_std,macro_f1,macro_f1_std,binary_f1,binary_f1_std,pr_auc,pr_auc_std,note
0,BotHP (final epoch),"BotHP process.py: random 70/20/10, seed 0","follower, friend",BotHP test_mask (20% block),5,0.9101,0.0014,0.8833,0.0027,0.8273,0.0046,0.8869,0.0038,"argmax prediction (threshold 0.5, not tuned)"
1,BotHP (early-stopping epoch),"BotHP process.py: random 70/20/10, seed 0","follower, friend",BotHP test_mask (20% block),5,0.9102,0.0012,0.8840,0.0021,0.8289,0.0035,0.8878,0.0034,"argmax prediction (threshold 0.5, not tuned)"


## 4. Run B: our split (validation only)

In [ ]:
# Our split() needs only the bot labels; it is computed here with the team code and saved as indices.
sys.path.insert(0, TEAM)
from mgtab import split
y = torch.load(f"{RAW}/labels_bot.pt").numpy()
train, val, test = split(y)
assert (len(train), len(val), len(test)) == (7139, 1530, 1530) and y[val].sum() == 412
np.save(f"{RAW}/ours_train.npy", train); np.save(f"{RAW}/ours_val.npy", val)

s = open(f"{RAW}/process.py").read()
old = """                 train_mask=train_mask,
                 test_mask=val_mask,
                 val_mask=test_mask)"""
assert s.count(old) == 1
s = s.replace(old, """                 train_mask=sample_mask(np.load('ours_train.npy'), sample_number),
                 test_mask=sample_mask(np.load('ours_val.npy'), sample_number),
                 val_mask=sample_mask(np.load('ours_val.npy'), sample_number))""")
open(f"{RAW}/process_ours.py", "w").write(s)
%cd /content/BotHP/dataset/raw/MGTAB
!{PY} process_ours.py
run_bothp("our_split", "ours: split(), seed 42", "our validation (15%)")

/content/BotHP/dataset/raw/MGTAB
final_accuracy: 0.9016±0.0022
early-stopping_accuracy: 0.9088±0.0033
final_precision: 0.8365±0.0094
early-stopping_precision: 0.8394±0.0109
final_recall: 0.7888±0.0075
early-stopping_recall: 0.8180±0.0157
final_f1: 0.8119±0.0036
early-stopping_f1: 0.8284±0.0068
final_macro_f1: 0.8726±0.0025
early-stopping_macro_f1: 0.8831±0.0044
final_pr_auc: 0.8729±0.0061
early-stopping_pr_auc: 0.8783±0.0073



,method,split,relations,eval_set,runs,accuracy,accuracy_std,macro_f1,macro_f1_std,binary_f1,binary_f1_std,pr_auc,pr_auc_std,note
0,BotHP (final epoch),"BotHP process.py: random 70/20/10, seed 0","follower, friend",BotHP test_mask (20% block),5,0.9101,0.0014,0.8833,0.0027,0.8273,0.0046,0.8869,0.0038,"argmax prediction (threshold 0.5, not tuned)"
1,BotHP (early-stopping epoch),"BotHP process.py: random 70/20/10, seed 0","follower, friend",BotHP test_mask (20% block),5,0.9102,0.0012,0.8840,0.0021,0.8289,0.0035,0.8878,0.0034,"argmax prediction (threshold 0.5, not tuned)"
2,BotHP (final epoch),"ours: split(), seed 42","follower, friend",our validation (15%),5,0.9016,0.0022,0.8726,0.0025,0.8119,0.0036,0.8729,0.0061,"argmax prediction (threshold 0.5, not tuned)"
3,BotHP (early-stopping epoch),"ours: split(), seed 42","follower, friend",our validation (15%),5,0.9088,0.0033,0.8831,0.0044,0.8284,0.0068,0.8783,0.0073,"argmax prediction (threshold 0.5, not tuned)"


## 5. Save and download

`results/bothp.csv` plus both logs, as one zip. Also save this notebook with its outputs (File > Download > .ipynb).

In [ ]:
%cd /content/sna-content-moderation
df = pd.DataFrame(RESULTS)
df.to_csv("results/bothp.csv", index=False)
display(df.round(4))
!zip -q -r /content/bothp_outputs.zip results/bothp.csv results/logs
from google.colab import files
files.download("/content/bothp_outputs.zip")

/content/sna-content-moderation


,method,split,relations,eval_set,runs,accuracy,accuracy_std,macro_f1,macro_f1_std,binary_f1,binary_f1_std,pr_auc,pr_auc_std,note
0,BotHP (final epoch),"BotHP process.py: random 70/20/10, seed 0","follower, friend",BotHP test_mask (20% block),5,0.9101,0.0014,0.8833,0.0027,0.8273,0.0046,0.8869,0.0038,"argmax prediction (threshold 0.5, not tuned)"
1,BotHP (early-stopping epoch),"BotHP process.py: random 70/20/10, seed 0","follower, friend",BotHP test_mask (20% block),5,0.9102,0.0012,0.8840,0.0021,0.8289,0.0035,0.8878,0.0034,"argmax prediction (threshold 0.5, not tuned)"
2,BotHP (final epoch),"ours: split(), seed 42","follower, friend",our validation (15%),5,0.9016,0.0022,0.8726,0.0025,0.8119,0.0036,0.8729,0.0061,"argmax prediction (threshold 0.5, not tuned)"
3,BotHP (early-stopping epoch),"ours: split(), seed 42","follower, friend",our validation (15%),5,0.9088,0.0033,0.8831,0.0044,0.8284,0.0068,0.8783,0.0073,"argmax prediction (threshold 0.5, not tuned)"


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>